# 20 · The kernel: hub, fiber, run (paper §12)
Code: `VAPMIP/scaled_mind_eye_boxkite_kernel_monad.py` (`MindEyeBoxKiteKernel`, `hub_report`, `throw_fiber`, `run_boxkite`, `FiberThrow`), `VAPMIP/19D_boxkite_context_monad.py` (`GaugeEye.continuous_snapshot`, `GaugeHands.emit_steered`).

In [1]:
from pn import *
L = Ledger("20 · the kernel")
with quiet():
    import scaled_mind_eye_boxkite_kernel_monad as K
    kern = K.MindEyeBoxKiteKernel()
    hub = kern.hub_report()
print({k: (v if not isinstance(v, dict) else {kk: vv for kk, vv in list(v.items())[:3]}) for k, v in hub.items()})
L.check("the hub is confirmed before it is used (e₀ outside the geometry)", hub["e0_is_outside"]["e0_is_outside_the_geometry"])
L.check("hub fixed_point_weight is 1", abs(hub["hub_fixed_point_weight"] - 1.0) < 1e-12)

{'e0_is_outside': {'e0_is_a_pg32_point': False, 'e0_in_any_assessor': False, 'e0_associator_always_vanishes': True}, 'fixed_point_gluing': {'indices_in_no_assessor': [0, 8], 'e0_is_orphan': True, 'e8_is_orphan': True}, 'hub_fixed_point_weight': 1.0}


True

## Run a boxkite: one hub-anchored fiber per strut

In [2]:
with quiet(): throws = kern.run_boxkite([(s, 0.6) for s in kern.kite.struts])
for t in throws: print(" ", t)
L.check("seven fibers, one per strut", [t.strut for t in throws] == list(range(1, 8)))
L.check("fixed_point_weight at every landing is 1 − 0.6² = 0.64", all(abs(t.fixed_point_weight - 0.64) < 1e-12 for t in throws))
print("is_zero_divisor by strut:", {t.strut: t.is_zero_divisor for t in throws})
print("windspeed by strut:", {t.strut: round(t.a_matrix_basin_windspeed, 6) for t in throws})
print("sigma_self by strut:", {t.strut: round(t.sigma_self, 6) for t in throws}, "(the kernel's own red/blue split: red = k ≥ 8)")

  FiberThrow(strut=1, depth=0.600, hub_weight=0.640, ZD=False, windspeed=+0.000000)
  FiberThrow(strut=2, depth=0.600, hub_weight=0.640, ZD=False, windspeed=+0.000000)
  FiberThrow(strut=3, depth=0.600, hub_weight=0.640, ZD=False, windspeed=+0.000000)
  FiberThrow(strut=4, depth=0.600, hub_weight=0.640, ZD=False, windspeed=+12.000000)
  FiberThrow(strut=5, depth=0.600, hub_weight=0.640, ZD=False, windspeed=+12.000000)
  FiberThrow(strut=6, depth=0.600, hub_weight=0.640, ZD=False, windspeed=+12.000000)
  FiberThrow(strut=7, depth=0.600, hub_weight=0.640, ZD=False, windspeed=+12.000000)
is_zero_divisor by strut: {1: False, 2: False, 3: False, 4: False, 5: False, 6: False, 7: False}
windspeed by strut: {1: 0.0, 2: 0.0, 3: 0.0, 4: 12.0, 5: 12.0, 6: 12.0, 7: 12.0}
sigma_self by strut: {1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 5: 0.0, 6: 0.0, 7: 0.0} (the kernel's own red/blue split: red = k ≥ 8)


## Each throw starts from the hub, never from the previous landing

In [3]:
with quiet():
    a = kern.throw_fiber(3, 0.6, gamma_companion=False); kern.throw_fiber(5, 0.9, gamma_companion=False); b = kern.throw_fiber(3, 0.6, gamma_companion=False)
L.check("throwing strut 3 again after other throws gives the identical landing (no residue between fibers)", a.psi == b.psi and a.fixed_point_weight == b.fixed_point_weight)
try: kern.throw_fiber(1, 1.5)
except ValueError as e: print("depth 1.5 refused:", e); L.check("a depth outside [0,1] is refused", True, str(e))
else: L.check("a depth outside [0,1] is refused", False, "no error")

depth 1.5 refused: depth must be in [0,1], got 1.5


## Ledger

In [4]:
_n = L.show()

LEDGER — 20 · the kernel
  [PASS ] the hub is confirmed before it is used (e₀ outside the geometry)
  [PASS ] hub fixed_point_weight is 1
  [PASS ] seven fibers, one per strut
  [PASS ] fixed_point_weight at every landing is 1 − 0.6² = 0.64
  [PASS ] throwing strut 3 again after other throws gives the identical landing (no residue between fibers)
  [PASS ] a depth outside [0,1] is refused
          depth must be in [0,1], got 1.5
  -> PASS 6 · FAIL 0 · PROSE 0
